# From the next word to an answer: SmolLM2-360M and its tuned twin

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josterri/predicting-the-next-words-notebooks/blob/main/answer_addon_lab.ipynb)

SmolLM2-360M continues text. SmolLM2-360M-Instruct is the same network after two tuning steps,
fine-tuning on chat texts that end in an answer and preference tuning. Both receive the same
chat text when someone types "Who wrote Pride and Prejudice?". Every token written below is
the likeliest one, and every probability is computed exactly from the model's scores.

The first cell installs transformers 5.15.1; the second downloads both twins.

In [ ]:
%pip install -q transformers==5.15.1

In [2]:
import math

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_grad_enabled(False)
BASE = ("HuggingFaceTB/SmolLM2-360M", "f8027fd0eaeea54caa13c31d31b9fdc459c38b49")
TUNED = ("HuggingFaceTB/SmolLM2-360M-Instruct", "a10cc1512eabd3dde888204e902eca88bddb4951")
tok = AutoTokenizer.from_pretrained(TUNED[0], revision=TUNED[1])
base, tuned = (AutoModelForCausalLM.from_pretrained(name, revision=rev, dtype=torch.float32,
                                                    attn_implementation="eager").eval()
               for name, rev in (BASE, TUNED))
END = tok.convert_tokens_to_ids("<|im_end|>")


def chat(question):
    """The text the tuned twin receives when the question is typed."""
    return tok.apply_chat_template([{"role": "user", "content": question}], tokenize=False,
                                   add_generation_prompt=True)


def ids(text, special=True):
    return tok(text, return_tensors="pt", add_special_tokens=special).input_ids


def next_token(model, text):
    """The probability of every possible next token after the text: one pass."""
    return model(ids(text)).logits[0, -1].softmax(-1)


def show(pairs):
    return ", ".join("%r %.3f" % (tok.decode([int(i)]), float(p)) for i, p in pairs)


def whole(model, prefix, continuation):
    """log10 of the probability of a whole continuation: the chain rule over its tokens."""
    a, b = ids(prefix), ids(continuation, special=False)
    lp = model(torch.cat([a, b], 1)).logits[0].log_softmax(-1)
    return sum(lp[a.shape[1] + j - 1, b[0, j]].item() for j in range(b.shape[1])) / math.log(10)


print("blocks %d, width %d, heads %d, vocabulary %d" % (tuned.config.num_hidden_layers,
      tuned.config.hidden_size, tuned.config.num_attention_heads, tuned.config.vocab_size))

blocks 32, width 960, heads 15, vocabulary 49152


## 1. The typed question becomes a chat text

The template wraps the typed question in a system sentence the user never sees, role markers
and an opened assistant turn. Count the tokens of each turn. To count another question, run
`chat("Who wrote Emma?")` in a new cell; steps 2 to 4 use `QUESTION` as set here.

In [3]:
QUESTION = "Who wrote Pride and Prejudice?"
text = chat(QUESTION)
print(text)
pieces = [tok.decode([int(t)]) for t in ids(text)[0]]
starts = [i for i, p in enumerate(pieces) if p == "<|im_start|>"]
print("tokens: %d = system %d + user %d + assistant %d" % (
    len(pieces), starts[1] - starts[0], starts[2] - starts[1], len(pieces) - starts[2]))
print("question tokens: %d" % len(ids(QUESTION, special=False)[0]))

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Who wrote Pride and Prejudice?<|im_end|>
<|im_start|>assistant

tokens: 37 = system 21 + user 12 + assistant 4
question tokens: 7


## 2. One pass gives every token a probability

One pass turns the chat text into a probability for every token of the vocabulary. Which
first token does each twin rank highest, and what share do the tuned
twin's five likeliest tokens hold together?

In [4]:
for name, model in (("tuned", tuned), ("base", base)):
    p = next_token(model, chat(QUESTION))
    v, i = p.topk(5)
    print("%-5s %s; together %.3f" % (name, show(zip(i, v)), float(v.sum())))

tuned 'The' 0.205, 'Pr' 0.122, 'I' 0.101, 'Based' 0.064, 'H' 0.056; together 0.548


base  'Who' 0.419, 'What' 0.121, 'You' 0.040, 'How' 0.035, 'Ass' 0.019; together 0.633


## 3. Three continuations, two twins, two formats

A is another question, B the true answer, C the same sentence with a false author. Each
continuation's probability is the product of its token probabilities. In the chat text the
base twin ranks A first. After "Question: ... Answer:" the same base twin ranks B first: the
fact was stored in pretraining, and tuning taught the chat format.

In [5]:
CONTINUATIONS = {"A": "Who wrote The Great Gatsby?",
                 "B": "The novel Pride and Prejudice was written by Jane Austen.",
                 "C": "The novel Pride and Prejudice was written by Charlotte Brontë."}
FORMATS = {"chat text": (chat(QUESTION), ""),
           "Question: ... Answer:": ("Question: %s\nAnswer:" % QUESTION, " ")}
for fmt, (prefix, lead) in FORMATS.items():
    for name, model in (("base", base), ("tuned", tuned)):
        s = {k: whole(model, prefix, lead + c) for k, c in CONTINUATIONS.items()}
        ratio = 10 ** (s["B"] - s["C"])
        print("%-22s %-5s log10 A %.2f, B %.2f, C %.2f; B/C %s" % (
            fmt, name, s["A"], s["B"], s["C"], "%.1f" % ratio if ratio < 100 else "{:,.0f}".format(ratio)))

chat text              base  log10 A -2.67, B -11.83, C -12.73; B/C 7.9


chat text              tuned log10 A -6.10, B -1.81, C -5.34; B/C 3,405
Question: ... Answer:  base  log10 A -6.85, B -5.64, C -8.22; B/C 387


Question: ... Answer:  tuned log10 A -7.84, B -4.18, C -7.36; B/C 1,534


## 4. The answer grows by one token per pass

Each pass appends the likeliest token and runs again on the longer text, until the
end-of-turn token. How many passes write the answer, and which steps are the least certain?

In [6]:
def greedy_steps(model, text, limit=30):
    """One pass per token: the likeliest token, its probability, then the text grows by it."""
    x, out = ids(text), []
    for _ in range(limit):
        p = model(x).logits[0, -1].softmax(-1)
        i = int(p.argmax())
        out.append((tok.decode([i]), float(p[i]), p))
        x = torch.cat([x, torch.tensor([[i]])], 1)
        if i == END:
            break
    return out


steps = greedy_steps(tuned, chat(QUESTION))
print(", ".join("%r %.3f" % (t, q) for t, q, _ in steps))
print("passes: %d" % len(steps))
print("product of the sentence %.4f, with the end token %.4f" % (
    math.prod(q for _, q, _ in steps[:-1]), math.prod(q for _, q, _ in steps)))

'The' 0.205, ' novel' 0.457, ' Pride' 0.807, ' and' 0.999, ' Pre' 1.000, 'judice' 1.000, ' was' 0.456, ' written' 0.876, ' by' 0.984, ' Jane' 0.684, ' Austen' 0.997, '.' 0.764, '<|im_end|>' 0.670
passes: 13
product of the sentence 0.0155, with the end token 0.0104


## 5. A novel with no record

No novel called The Lanterns of Chur exists. The tuned twin still writes an author. Compare
the probability of the name with the probabilities of the words around it. The first token
I, less likely than The, opens a refusal.

In [7]:
INVENTED = "Who wrote the novel The Lanterns of Chur?"
p = next_token(tuned, chat(INVENTED))
v, i = p.topk(3)
print("first token: %s" % show(zip(i, v)))
steps = greedy_steps(tuned, chat(INVENTED))
print("".join(t for t, _, _ in steps))
for t, q, dist in steps:
    if t == " Gabriel":
        v, i = dist.topk(3)
        print("the name: %s" % show(zip(i, v)))
print("after I: I" + "".join(t for t, _, _ in greedy_steps(tuned, chat(INVENTED) + "I")))

first token: 'The' 0.587, 'I' 0.078, 'Based' 0.072


The novel The Lanterns of Chur was written by the renowned author, Gabriel García Márquez.<|im_end|>
the name: ' Gabriel' 0.028, ' Thomas' 0.021, ' John' 0.020


after I: I'm sorry for the confusion, but as a text-based model, I don't have the ability to access or access information outside of my training data


## Try

- Change `chat(QUESTION)` in step 2 to `chat("Who wrote Jane Eyre?")` and rerun the cell.
  Does the base twin still put Who first?
- In step 5, ask about a second invented title. Which author does the tuned twin name, and
  at what probability?